# D4-v7 Batch 1 Analysis: FoundOpt + RL (MAIN vs B4)

4 runs completed (25 iters each):
- FoundOpt MAIN (full pipeline: buffer + critique + RL)
- FoundOpt B4 (buffer + critique, no RL)
- RL MAIN
- RL B4

Key questions:
1. Does MAIN beat B4 on fresh_reward_mean (BoN=1 proxy)?
2. How do buffer_max curves compare?
3. Which signals improve most with training?
4. What's the hard gate pass rate?

In [3]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

RUN_DIR = Path('../runs')
DATE = '2026_04_20'

def load_metrics(goal, condition):
    path = RUN_DIR / f'{DATE}_d4v7_{goal}_{condition}' / 'metrics.jsonl'
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def load_buffer(goal, condition):
    path = RUN_DIR / f'{DATE}_d4v7_{goal}_{condition}' / 'buffer.jsonl'
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

runs = {
    'FO_MAIN': ('01_foundopt', 'MAIN'),
    'FO_B4':   ('01_foundopt', 'B4'),
    'RL_MAIN': ('02_foundational_rl', 'MAIN'),
    'RL_B4':   ('02_foundational_rl', 'B4'),
}

metrics = {k: load_metrics(*v) for k, v in runs.items()}
buffers = {k: load_buffer(*v) for k, v in runs.items()}

colors = {'FO_MAIN': '#2196F3', 'FO_B4': '#90CAF9', 'RL_MAIN': '#E91E63', 'RL_B4': '#F48FB1'}
labels = {'FO_MAIN': 'FoundOpt MAIN', 'FO_B4': 'FoundOpt B4', 'RL_MAIN': 'RL MAIN', 'RL_B4': 'RL B4'}
styles = {'FO_MAIN': '-', 'FO_B4': '--', 'RL_MAIN': '-', 'RL_B4': '--'}

loaded_str = ", ".join(f"{k}: {len(v)} iters" for k, v in metrics.items())
print(f"Loaded: {loaded_str}")

Loaded: FO_MAIN: 25 iters, FO_B4: 25 iters, RL_MAIN: 25 iters, RL_B4: 25 iters


## 1. Training Curves: buffer_max + fresh_reward_mean

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {'FO_MAIN': '#2196F3', 'FO_B4': '#90CAF9', 'RL_MAIN': '#E91E63', 'RL_B4': '#F48FB1'}
labels = {'FO_MAIN': 'FoundOpt MAIN', 'FO_B4': 'FoundOpt B4', 'RL_MAIN': 'RL MAIN', 'RL_B4': 'RL B4'}
styles = {'FO_MAIN': '-', 'FO_B4': '--', 'RL_MAIN': '-', 'RL_B4': '--'}

# Plot 1: buffer_max
ax = axes[0]
for k in runs:
    iters = [m['iter'] for m in metrics[k]]
    vals = [m.get('reward/buffer_max', 0) for m in metrics[k]]
    ax.plot(iters, vals, styles[k], color=colors[k], label=labels[k], linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Reward')
ax.set_title('buffer_max (best plan ever seen)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 2: fresh_reward_mean
ax = axes[1]
for k in runs:
    iters = [m['iter'] for m in metrics[k]]
    vals = [m.get('fresh/reward_mean', 0) for m in metrics[k]]
    ax.plot(iters, vals, styles[k], color=colors[k], label=labels[k], linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Reward')
ax.set_title('fresh_reward_mean (BoN=1 proxy)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 3: revision success rate
ax = axes[2]
for k in runs:
    iters = [m['iter'] for m in metrics[k]]
    n_pos = [m.get('revise/n_positive', 0) for m in metrics[k]]
    n_rev = [m.get('revise/count', 0) for m in metrics[k]]
    rate = [p/r if r > 0 else 0 for p, r in zip(n_pos, n_rev)]
    ax.plot(iters, rate, styles[k], color=colors[k], label=labels[k], linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Rate')
ax.set_title('Revision Success Rate (positive delta)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Per-Signal Evolution

In [ ]:
signals = ['G1_problem_specificity', 'G2_specific_aims', 'G3_technical_evidence',
           'G4_focus', 'G5_gap_identification', 'G6_reasoning_depth',
           'G8_deliverable_clarity', 'G9_scope_feasibility', 'G10_approach_coverage',
           'G11_evidence_rigor', 'G12_formalism', 'G13_risk_awareness']
short_names = [s.split('_', 1)[1] for s in signals]

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for idx, (sig, sname) in enumerate(zip(signals, short_names)):
    ax = axes[idx]
    for k in runs:
        iters = [m['iter'] for m in metrics[k]]
        vals = [m.get(f'signal/{sig}/mean', 0) for m in metrics[k]]
        ax.plot(iters, vals, styles[k], color=colors[k], label=labels[k] if idx == 0 else '', linewidth=1.5)
    ax.set_title(sname, fontsize=10)
    ax.set_ylim(-0.1, 5.5)
    ax.grid(alpha=0.2)
    if idx >= 8:
        ax.set_xlabel('Iteration')

axes[0].legend(fontsize=7, loc='lower right')
plt.suptitle('Per-Signal Mean Score (1-5) by Iteration', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Hard Gate Pass Rate

In [6]:
print(f'{"Run":12s} | {"Total":>6s} | {"HG Pass":>7s} | {"Rate":>6s} | {"GCM fail":>8s}')
print('-' * 55)
for k in runs:
    buf = buffers[k]
    total = len(buf)
    passed = sum(1 for e in buf if e.get('hard_gate_passed', True))
    gcm_fail = sum(1 for e in buf if e.get('hard_gate_failed_name') == 'goal_contrast_margin')
    rate = passed / total if total > 0 else 0
    print(f'{labels[k]:12s} | {total:6d} | {passed:7d} | {rate:6.1%} | {gcm_fail:8d}')

Run          |  Total | HG Pass |   Rate | GCM fail
-------------------------------------------------------
FoundOpt MAIN |    195 |     180 |  92.3% |       15
FoundOpt B4  |    195 |     183 |  93.8% |       12
RL MAIN      |    196 |     193 |  98.5% |        3
RL B4        |    195 |     190 |  97.4% |        5


## 4. Final Comparison Table

In [7]:
print(f'{"":15s} | {"buf_max":>8s} | {"fresh_r":>8s} | {"fresh_r":>8s} | {"rev+":>5s}')
print(f'{"Run":15s} | {"final":>8s} | {"final":>8s} | {"avg(10-24)":>8s} | {"rate":>5s}')
print('-' * 60)

for k in runs:
    last = metrics[k][-1]
    bm = last.get('reward/buffer_max', 0)
    fr = last.get('fresh/reward_mean', 0)
    # Average fresh_r over last 15 iters for more stable estimate
    late_fr = [m.get('fresh/reward_mean', 0) for m in metrics[k] if m['iter'] >= 10]
    late_fr = [v for v in late_fr if v > 0]  # exclude failed iters
    avg_late = np.mean(late_fr) if late_fr else 0
    # Revision success rate
    total_rev = sum(m.get('revise/count', 0) for m in metrics[k])
    total_pos = sum(m.get('revise/n_positive', 0) for m in metrics[k])
    rev_rate = total_pos / total_rev if total_rev > 0 else 0
    print(f'{labels[k]:15s} | {bm:8.3f} | {fr:8.3f} | {avg_late:8.3f} | {rev_rate:5.1%}')

                |  buf_max |  fresh_r |  fresh_r |  rev+
Run             |    final |    final | avg(10-24) |  rate
------------------------------------------------------------
FoundOpt MAIN   |    0.970 |    0.886 |    0.803 | 41.1%
FoundOpt B4     |    1.000 |    0.845 |    0.806 | 30.5%
RL MAIN         |    0.940 |    0.835 |    0.836 | 39.6%
RL B4           |    0.945 |    0.860 |    0.825 | 34.4%


## 5. Signal Scores: Initial vs Final (Iter 0 vs Iter 24)

In [8]:
print('Signal improvement from iter 0 → iter 24 (mean across MAIN runs)\n')
print(f'{"Signal":25s} | {"Iter 0":>6s} | {"Iter 24":>7s} | {"Δ":>6s}')
print('-' * 52)

for sig, sname in zip(signals, short_names):
    vals_0 = []
    vals_24 = []
    for k in ['FO_MAIN', 'RL_MAIN']:
        v0 = metrics[k][0].get(f'signal/{sig}/mean', 0)
        v24 = metrics[k][-1].get(f'signal/{sig}/mean', 0)
        if v0 > 0: vals_0.append(v0)
        if v24 > 0: vals_24.append(v24)
    m0 = np.mean(vals_0) if vals_0 else 0
    m24 = np.mean(vals_24) if vals_24 else 0
    delta = m24 - m0
    marker = '⬆' if delta > 0.5 else ('⬇' if delta < -0.5 else '—')
    print(f'{sname:25s} | {m0:6.1f} | {m24:7.1f} | {delta:+6.1f} {marker}')

Signal improvement from iter 0 → iter 24 (mean across MAIN runs)

Signal                    | Iter 0 | Iter 24 |      Δ
----------------------------------------------------
problem_specificity       |    5.0 |     5.0 |   +0.0 —
specific_aims             |    5.0 |     5.0 |   +0.0 —
technical_evidence        |    3.6 |     5.0 |   +1.4 ⬆
focus                     |    2.9 |     3.0 |   +0.1 —
gap_identification        |    4.5 |     5.0 |   +0.5 —
reasoning_depth           |    2.1 |     4.3 |   +2.2 ⬆
deliverable_clarity       |    3.5 |     4.9 |   +1.3 ⬆
scope_feasibility         |    1.7 |     4.1 |   +2.4 ⬆
approach_coverage         |    4.9 |     5.0 |   +0.1 —
evidence_rigor            |    1.2 |     4.5 |   +3.3 ⬆
formalism                 |    1.0 |     4.8 |   +3.8 ⬆
risk_awareness            |    1.9 |     4.4 |   +2.5 ⬆


## 6. MAIN vs B4: BoN=1 Gap Over Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (goal_name, main_k, b4_k) in enumerate([('FoundOpt', 'FO_MAIN', 'FO_B4'),
                                                    ('Foundational RL', 'RL_MAIN', 'RL_B4')]):
    ax = axes[idx]
    iters_m = [m['iter'] for m in metrics[main_k]]
    iters_b = [m['iter'] for m in metrics[b4_k]]
    fr_m = [m.get('fresh/reward_mean', 0) for m in metrics[main_k]]
    fr_b = [m.get('fresh/reward_mean', 0) for m in metrics[b4_k]]
    
    ax.plot(iters_m, fr_m, '-', color='#2196F3', label='MAIN', linewidth=2)
    ax.plot(iters_b, fr_b, '--', color='#FF9800', label='B4 (no RL)', linewidth=2)
    ax.fill_between(iters_m, fr_b[:len(iters_m)], fr_m, 
                     where=[m > b for m, b in zip(fr_m, fr_b[:len(iters_m)])],
                     alpha=0.2, color='#2196F3', label='MAIN > B4')
    ax.fill_between(iters_m, fr_b[:len(iters_m)], fr_m,
                     where=[m < b for m, b in zip(fr_m, fr_b[:len(iters_m)])],
                     alpha=0.2, color='#FF9800', label='B4 > MAIN')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('fresh_reward_mean')
    ax.set_title(f'{goal_name}: MAIN vs B4 (BoN=1)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Summary

In [10]:
# Compute final summary stats
summary = {}
for k in runs:
    last = metrics[k][-1]
    late_fr = [m.get('fresh/reward_mean', 0) for m in metrics[k] if m['iter'] >= 10]
    late_fr = [v for v in late_fr if v > 0]
    buf = buffers[k]
    passed = sum(1 for e in buf if e.get('hard_gate_passed', True))
    summary[k] = {
        'buffer_max': last.get('reward/buffer_max', 0),
        'fresh_r_final': last.get('fresh/reward_mean', 0),
        'fresh_r_avg_late': np.mean(late_fr) if late_fr else 0,
        'hg_pass_rate': passed / len(buf) if buf else 0,
    }

print('=== D4-v7 Batch 1 Summary ===')
print()
print('Key finding: RL improves POLICY QUALITY (fresh_reward_mean) but not buffer_max.')
print('This matches D3 v9: RL value = exploration, not exploitation.\n')

for goal, mk, bk in [('FoundOpt', 'FO_MAIN', 'FO_B4'), ('RL', 'RL_MAIN', 'RL_B4')]:
    m = summary[mk]
    b = summary[bk]
    print(f'{goal}:')
    print(f'  buffer_max:  MAIN={m["buffer_max"]:.3f}  B4={b["buffer_max"]:.3f}  Δ={m["buffer_max"]-b["buffer_max"]:+.3f}')
    print(f'  BoN=1 (avg): MAIN={m["fresh_r_avg_late"]:.3f}  B4={b["fresh_r_avg_late"]:.3f}  Δ={m["fresh_r_avg_late"]-b["fresh_r_avg_late"]:+.3f}')
    print(f'  HG pass:     MAIN={m["hg_pass_rate"]:.0%}  B4={b["hg_pass_rate"]:.0%}')
    print()

=== D4-v7 Batch 1 Summary ===

Key finding: RL improves POLICY QUALITY (fresh_reward_mean) but not buffer_max.
This matches D3 v9: RL value = exploration, not exploitation.

FoundOpt:
  buffer_max:  MAIN=0.970  B4=1.000  Δ=-0.030
  BoN=1 (avg): MAIN=0.803  B4=0.806  Δ=-0.003
  HG pass:     MAIN=92%  B4=94%

RL:
  buffer_max:  MAIN=0.940  B4=0.945  Δ=-0.005
  BoN=1 (avg): MAIN=0.836  B4=0.825  Δ=+0.011
  HG pass:     MAIN=98%  B4=97%

